# 10. Futility-анализ (ранняя остановка по бесперспективности)


## 0. Мотивация: зачем останавливаться по бесперспективности

В прошлой лекции мы останавливали эксперимент **раньше**, когда эффект уже доказан (efficacy-остановка). Теперь — «обратная сторона»: как остановить тест, когда эффекта, скорее всего, **нет или он меньше MDE**.

Продуктовая логика проста: трафик и время — ограниченный ресурс. Продолжать «мёртвый» тест — значит жечь трафик и задерживать другие эксперименты (opportunity cost).

Ключевой вопрос лекции:

> Если на промежуточном взгляде эффект выглядит слабее MDE, есть ли смысл тратить оставшийся трафик?

Ответ даёт **futility-анализ** — набор правил ранней остановки по бесперспективности. Мы построим его на той же математике, что и sequential testing: стандартизованная статистика $Z_t$ и информационная доля $t$.


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

SEED = 12345
rng = np.random.default_rng(SEED)

C_FUT = "#d95f02"   # оранжевый — futility
C_EFF = "#1b9e77"   # зелёный — efficacy
C_CONT = "#999999"  # серый — продолжаем


In [ ]:
@dataclass(frozen=True)
class Design:
    """Параметры сквозного примера: A/B-тест непрерывной метрики."""
    alpha: float = 0.05        # односторонняя ошибка I рода
    beta: float = 0.20         # ошибка II рода (power = 0.80)
    sigma: float = 100.0       # СКО метрики (средний чек, руб.)
    mu0: float = 500.0         # среднее в контроле
    mde: float = 10.0          # абсолютный MDE (руб.)
    K: int = 5
    t: tuple = (0.30, 0.50, 0.70, 0.85, 1.0)  # доли информации (взгляды)

    @property
    def delta(self) -> float:
        """Стандартизованный MDE: delta = mde / sigma."""
        return self.mde / self.sigma

    @property
    def theta_mde(self) -> float:
        """Дрейф (нецентральность) под MDE: theta = z_{1-alpha} + z_{1-beta}."""
        return norm.ppf(1 - self.alpha) + norm.ppf(1 - self.beta)

    @property
    def i_max(self) -> float:
        """Максимальная информация: I_max = (theta_mde / delta)^2."""
        return (self.theta_mde / self.delta) ** 2

    @property
    def n_max(self) -> float:
        """Максимальный размер выборки на группу (для двухвыборочного z-теста)."""
        return 2 * self.sigma ** 2 * self.i_max


design = Design()
design


### 0.1. «Цена» продолжения теста

Зафиксируем сквозной пример: метрика — **средний чек** (непрерывная, $\sigma = 100$ руб.), контроль $\mu_0 = 500$ руб., хотим поймать подъём $MDE = 10$ руб. с мощностью 80% при одностороннем $\alpha = 5\%$.

Ниже — сколько трафика «стоит» один полный тест и сколько мы экономим, если останавливаемся раньше.


In [ ]:
n = int(np.ceil(design.n_max))
print(f"Стандартизованный MDE: delta = {design.delta:.3f}")
print(f"Дрейф под MDE:         theta_mde = {design.theta_mde:.3f}")
print(f"Максимальный размер на группу: n_max = {n:,} пользователей")
print(f"Суммарный трафик полного теста: {2*n:,} пользователей")

cost = pd.DataFrame({
    "Момент остановки": ["25%", "50%", "75%", "100%"],
    "Доля трафика": [0.25, 0.50, 0.75, 1.00],
    "Пользователей": [int(2*n*x) for x in (0.25, 0.50, 0.75, 1.00)],
})
cost["Сэкономлено vs 100%"] = (2*n - cost["Пользователей"]).astype(int)
cost


## 1. Аппарат: информация, дрейф и B-value (мостик к прошлой лекции)

Вся futility-механика живёт на одной оси — стандартизованной статистике $Z_t$ и доле информации $t = I_t / I_{max}$. Короткое напоминание (без повтора SPRT/AVI).

Для двухвыборочного теста непрерывной метрики с равными группами статистика в момент $t$:

$$Z_t = \frac{\bar X_B - \bar X_A}{\sigma \sqrt{2/n_t}} \;\sim\; \mathcal{N}\big(\theta\sqrt{t},\;1\big),$$

где дрейф $\theta = \delta\sqrt{I_{max}}$ — это «нецентральность» финального анализа, а приращения процесса **независимы**:

$$\mathrm{Cov}(Z_s, Z_t) = \sqrt{s/t}, \qquad s \le t.$$

Отсюда удобная «броуновская» форма $B(t) = \sqrt{t}\, Z_t$, у которой $B(t) \sim \mathcal{N}(\theta t,\; t)$ — с ней проще считать вероятности пересечения границ.


In [ ]:
def z_stat(xA: np.ndarray, xB: np.ndarray, sigma: float) -> float:
    """Двухвыборочный z-тест непрерывной метрики (известная дисперсия)."""
    nA, nB = xA.size, xB.size
    se = sigma * np.sqrt(1 / nA + 1 / nB)
    return (xB.mean() - xA.mean()) / se


def simulate_b(rng, theta: float, t: np.ndarray, n_sim: int = 200) -> np.ndarray:
    """Траектории B(t) = sqrt(t) * Z_t как броуновское движение с дрейфом theta."""
    dt = np.diff(np.concatenate(([0.0], t)))
    inc = rng.normal(loc=theta * dt, scale=np.sqrt(dt), size=(n_sim, len(t)))
    return np.cumsum(inc, axis=1)


In [ ]:
# Проверка: Z_t ~ N(theta*sqrt(t), 1)
t = np.asarray(design.t)
Z = simulate_b(rng, design.theta_mde, t, 20000) / np.sqrt(t)
print("Среднее Z_t под H1:", np.round(Z.mean(axis=0), 3))
print("Ожидаемое theta*sqrt(t):", np.round(design.theta_mde * np.sqrt(t), 3))
print("SD Z_t (должно быть ~1):", np.round(Z.std(axis=0), 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, theta, title in [(axes[0], 0.0, "H0: эффекта нет"),
                          (axes[1], design.theta_mde, "H1: эффект = MDE")]:
    B = simulate_b(rng, theta, t, 200)
    ax.plot(t, B.T, color=C_CONT, alpha=0.25, lw=1)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("доля информации t")
axes[0].set_ylabel("B(t) = sqrt(t) * Z_t")
fig.suptitle("Броуновское движение с дрейфом: под H0 траектории топчутся у нуля, под H1 — уходят вверх")
fig.tight_layout()
plt.show()


## 2. Conditional Power (CP): условная вероятность «добить» тест

Главный частотный инструмент futility. **Conditional power** — вероятность в конце отвергнуть $H_0$ при условии текущих данных $Z_t = z_t$ и при **заданном** предположении об эффекте $\theta$:

$$\mathrm{CP}(\theta) = P_\theta\big(Z_1 > z_{1-\alpha} \mid Z_t = z_t\big)
= 1 - \Phi\!\left(\frac{z_{1-\alpha} - \sqrt{t}\, z_t - \theta(1-t)}{\sqrt{1-t}}\right).$$

Правило futility: **если $\mathrm{CP} < \gamma$ — останавливаемся** (типично $\gamma \in [0.10, 0.20]$).

Ключевой нюанс — **какое $\theta$ подставить**:
- $\theta = 0$ (под $H_0$) — консервативно;
- $\theta = \hat\theta_t = z_t/\sqrt t$ (текущий тренд) — оптимистично и шумно на ранних взглядах;
- $\theta = \theta_{MDE}$ (дизайновый эффект) — стандартный ориентир.


In [ ]:
def conditional_power(z_t: float, t: float, theta: float, z_alpha: float = 1.6448536) -> float:
    """CP: вероятность Z_1 > z_alpha при данных z_t и истинном дрейфе theta."""
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - theta * (1 - t)) / np.sqrt(1 - t))


In [ ]:
z_grid = np.linspace(-3, 3, 300)
tt = 0.5
z_alpha = norm.ppf(1 - design.alpha)

fig, ax = plt.subplots(figsize=(7.5, 4))
for theta, label, color in [(0.0, "CP под H0", C_CONT),
                             (design.theta_mde, "CP под MDE", C_EFF),
                             (z_grid / np.sqrt(tt), "CP под трендом", C_FUT)]:
    cp = conditional_power(z_grid, tt, theta, z_alpha)
    ax.plot(z_grid, cp, label=label, color=color, lw=2)
ax.axhline(0.15, ls="--", color="black", lw=1, label="порог futility γ=0.15")
ax.set_xlabel("наблюдаемая статистика z_t")
ax.set_ylabel("conditional power")
ax.set_title(f"CP при t = {tt}: выбор θ кардинально меняет ответ")
ax.legend()
plt.show()


### 2.1. Карта CP и порог futility

Геометрически CP — это «карта шанса дойти до значимости». На ранних взглядах карта почти плоская (впереди много данных), к концу — «заостряется» (судьба почти решена). Контур $\mathrm{CP} = \gamma$ и есть правило остановки.


In [ ]:
T, Zz = np.meshgrid(np.linspace(0.05, 0.95, 100), np.linspace(-3, 3, 200))
CP = conditional_power(Zz, T, design.theta_mde, z_alpha)

fig, ax = plt.subplots(figsize=(7.5, 5))
cs = ax.contourf(T, Zz, CP, levels=20, cmap="viridis")
ax.contour(T, Zz, CP, levels=[0.15], colors="red", linewidths=2)
ax.set_xlabel("доля информации t")
ax.set_ylabel("наблюдаемая статистика z_t")
ax.set_title("CP под MDE: контур — правило futility (CP = 0.15)")
fig.colorbar(cs, label="conditional power")
plt.show()


In [ ]:
def plot_cp_interactive():
    def _plot(t, z_t, theta, alpha):
        za = norm.ppf(1 - alpha)
        cp = conditional_power(z_t, t, theta, za)
        fig, ax = plt.subplots(figsize=(7, 2.6))
        ax.barh(0, cp, color=C_FUT if cp < 0.15 else C_EFF)
        ax.set_xlim(0, 1)
        ax.set_yticks([])
        ax.axvline(0.15, ls="--", color="black")
        ax.set_title(f"CP = {cp:.3f}  →  {'СТОП (futility)' if cp < 0.15 else 'продолжаем'}")
        plt.show()
    interact(_plot,
             t=FloatSlider(value=0.3, min=0.05, max=0.95, step=0.05, description="t"),
             z_t=FloatSlider(value=0.5, min=-3, max=3, step=0.05, description="z_t"),
             theta=FloatSlider(value=design.theta_mde, min=0, max=4, step=0.1, description="θ"),
             alpha=FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description="α"))
plot_cp_interactive()


## 3. Predictive Power (PPoS): усредняем CP по неопределённости эффекта

Претензия к CP — жёсткая фиксация $\theta$. **Predictive power / Probability of Success (PPoS)** снимает её, усредняя CP по апостериорной неопределённости эффекта:

$$\mathrm{PPoS} = \int \mathrm{CP}(\theta)\,\pi(\theta \mid D_t)\,d\theta.$$

При нормальном априорном $\theta \sim \mathcal{N}(\theta_0, \tau^2)$ апостериор тоже нормальный, и интеграл берётся в замкнутой форме (Spiegelhalter–Freedman–Parmar):

$$\mathrm{PPoS} = 1 - \Phi\!\left(\frac{z_{1-\alpha} - \sqrt{t}\, z_t - \mu_p(1-t)}
{\sqrt{(1-t) + (1-t)^2 \sigma_p^2}}\right),$$

где $1/\sigma_p^2 = 1/\tau^2 + t$, $\;\mu_p = \sigma_p^2\left(\theta_0/\tau^2 + \sqrt{t}\, z_t\right)$. CP — частный случай PPoS с точечным (вырожденным) априорным.


In [ ]:
def ppos(z_t: float, t: float, theta0: float, tau: float, z_alpha: float = 1.6448536) -> float:
    """Predictive probability of success (нормальный prior, замкнутая форма)."""
    sigma2 = 1.0 / (1.0 / tau**2 + t)
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - mu * (1 - t))
                          / np.sqrt((1 - t) + (1 - t)**2 * sigma2))


In [ ]:
z_t, tt = 0.4, 0.5
theta_grid = np.linspace(-0.5, 4, 400)
theta0, tau = 0.0, 0.8          # скептический prior
sigma2 = 1.0 / (1.0 / tau**2 + tt)
mu = sigma2 * (theta0 / tau**2 + np.sqrt(tt) * z_t)

cp_theta = conditional_power(z_t, tt, theta_grid, z_alpha)
post = norm.pdf(theta_grid, loc=mu, scale=np.sqrt(sigma2))
pp = ppos(z_t, tt, theta0, tau, z_alpha)
cp_trend = conditional_power(z_t, tt, z_t / np.sqrt(tt), z_alpha)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(theta_grid, post, color=C_CONT)
axes[0].axvline(mu, color="black", ls="--", lw=1, label="апостериорное среднее")
axes[0].set_title("Апостериор эффекта θ | данные")
axes[0].set_xlabel("θ"); axes[0].set_ylabel("плотность")
axes[0].legend()

axes[1].plot(theta_grid, cp_theta, color=C_EFF, lw=2, label="CP(θ)")
axes[1].fill_between(theta_grid, cp_theta * post, alpha=0.3, color=C_FUT)
axes[1].axhline(pp, color=C_FUT, ls="--", label=f"PPoS = {pp:.3f}")
axes[1].axhline(cp_trend, color=C_CONT, ls="--", label=f"CP(тренд) = {cp_trend:.3f}")
axes[1].set_title("PPoS = среднее CP(θ) по апостериору")
axes[1].set_xlabel("θ"); axes[1].legend()
fig.suptitle("PPoS «притягивается» к массе апостериора у нуля → честнее CP(тренд)")
fig.tight_layout()
plt.show()


In [ ]:
def plot_ppos_interactive():
    def _plot(t, z_t, theta0, tau):
        pp = ppos(z_t, t, theta0, tau, z_alpha)
        cpt = conditional_power(z_t, t, z_t / np.sqrt(t), z_alpha)
        fig, ax = plt.subplots(figsize=(7, 2.4))
        ax.barh([1, 0], [pp, cpt], color=[C_FUT, C_CONT])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["CP(тренд)", "PPoS"])
        ax.set_xlim(0, 1); ax.axvline(0.15, ls="--", color="black")
        ax.set_title(f"PPoS = {pp:.3f}   vs   CP(тренд) = {cpt:.3f}")
        plt.show()
    interact(_plot,
             t=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="t"),
             z_t=FloatSlider(value=0.4, min=-3, max=3, step=0.05, description="z_t"),
             theta0=FloatSlider(value=0.0, min=-1, max=3, step=0.1, description="prior θ0"),
             tau=FloatSlider(value=0.8, min=0.05, max=3, step=0.05, description="prior τ"))
plot_ppos_interactive()


## 4. Stochastic Curtailment: общая рамка и перевод CP в границу

CP и PPoS — частные случаи **stochastic curtailment** (Lan–Simon–Halperin, 1982): останавливаем тест, когда условная вероятность «полезного» финального исхода при некоем предположении становится пренебрежимо малой.

Порог $\mathrm{CP} < \gamma$ можно перевести в **futility-границу на шкале $z$**: при фиксированных $t, \alpha, \gamma, \theta$ условие $\mathrm{CP} < \gamma$ эквивалентно $z_t < a_{CP}$, где $a_{CP}$ — решение уравнения $\mathrm{CP}(a_{CP}; t, \theta) = \gamma$.

Важно: это **данные-зависимая** граница (задаётся по ходу теста), в отличие от заранее фиксированной границы β-spending из следующего раздела.


In [ ]:
def cp_to_boundary(t: float, gamma: float, theta: float, z_alpha: float = 1.6448536) -> float:
    """z_t, при котором CP(z_t; t, theta) == gamma."""
    f = lambda z: conditional_power(z, t, theta, z_alpha) - gamma
    return brentq(f, -8, 8)


In [ ]:
tt_grid = np.linspace(0.05, 0.95, 200)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for theta, label, color in [(0.0, "граница по CP под H0", C_CONT),
                             (design.theta_mde, "граница по CP под MDE", C_FUT),
                             (design.theta_mde, "—", C_FUT)]:
    a = [cp_to_boundary(t_, 0.15, theta, z_alpha) for t_ in tt_grid]
    ax.plot(tt_grid, a, color=color, lw=2, label=label if color != C_FUT else None)
ax.plot(tt_grid, [cp_to_boundary(t_, 0.15, design.theta_mde, z_alpha) for t_ in tt_grid],
        color=C_FUT, lw=2, label="граница по CP под MDE")
ax.fill_between(tt_grid, -4, [cp_to_boundary(t_, 0.15, design.theta_mde, z_alpha) for t_ in tt_grid],
                color=C_FUT, alpha=0.1)
ax.set_xlabel("доля информации t"); ax.set_ylabel("futility-граница a(t) на шкале z")
ax.set_title("Futility-граница из порога CP=0.15: ниже кривой — остановка")
ax.legend(); plt.show()


## 5. β-spending: заранее фиксированная futility-граница

**β-spending — зеркало α-spending.** Как α-бюджет расходуется на efficacy-границу (контроль ошибки I рода), так β-бюджет — на futility-границу (контроль ошибки II рода). Обе живут на оси $t$.

- α-spending (Lan–DeMets): $α(t)$ неубывающая, $α(0)=0$, $α(1)=α$. Формы O'Brien–Fleming и Pocock.
- β-spending (Kim–DeMets / Pampallona–Tsiatis): $β(t)$ неубывающая, $β(0)=0$, $β(1)=β$. Степенное семейство $β(t)=β\,t^{\rho}$ ($\rho=3$ — OBF-подобная, $\rho=1$ — Pocock-подобная).

Границы считаются рекурсивно: на каждом взгляде efficacy-граница $c_k$ подбирается так, чтобы под $H_0$ «тратилось» $α(t_k)-α(t_{k-1})$, а futility-граница $a_k$ — чтобы под $H_1$ (θ=θ_MDE) тратилось $β(t_k)-β(t_{k-1})$. Ниже — честная численная рекурсия через плотность продолжения (свёртка независимых приращений B-процесса).


In [ ]:
def alpha_spending(t, alpha: float = 0.05, family: str = "obf") -> np.ndarray:
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return 2 - 2 * norm.cdf(norm.ppf(1 - alpha / 2) / np.sqrt(t))
    if family == "pocock":
        return alpha * np.log(1 + (np.e - 1) * t)
    raise ValueError(family)


def beta_spending(t, beta: float = 0.20, family: str = "obf") -> np.ndarray:
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return beta * t ** 3          # O'Brien–Fleming-подобная
    if family == "pocock":
        return beta * t               # Pocock-подобная
    raise ValueError(family)


In [ ]:
def _convolve(p: np.ndarray, grid: np.ndarray, loc: float, scale: float) -> np.ndarray:
    """p_next(x) = Σ_u p(u) * φ(x; u+loc, scale) * dx  (независимые приращения)."""
    dx = grid[1] - grid[0]
    K = norm.pdf(grid[:, None], loc=grid[None, :] + loc, scale=scale)
    return (K @ p) * dx


def _quantile(p: np.ndarray, grid: np.ndarray, target: float, tail: bool) -> float:
    dx = grid[1] - grid[0]
    cdf = np.cumsum(p) * dx
    vals = (cdf[-1] - cdf) if tail else cdf          # хвост или левая часть
    idx = int(np.argmin(np.abs(vals - target)))
    return grid[idx]


def group_sequential_boundaries(t, alpha=0.05, beta=0.20, theta=None,
                                fam_alpha="obf", fam_beta="obf",
                                grid_dx=0.01, n_sigma=6.0):
    """Односторонние efficacy (верхняя c) и futility (нижняя a) границы на шкале Z."""
    t = np.asarray(t, dtype=float)
    K = len(t)
    if theta is None:
        theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    dt = np.diff(np.concatenate(([0.0], t)))
    grid = np.arange(-n_sigma, n_sigma + grid_dx, grid_dx)

    d_alpha = np.diff(np.concatenate(([0.0], alpha_spending(t, alpha, fam_alpha))))
    d_beta = np.diff(np.concatenate(([0.0], beta_spending(t, beta, fam_beta))))

    c = np.empty(K); a = np.empty(K)
    p_eff = norm.pdf(grid, loc=0.0, scale=math.sqrt(dt[0]))            # под H0
    p_fut = norm.pdf(grid, loc=theta * dt[0], scale=math.sqrt(dt[0]))  # под H1
    b_prev, a_prev = np.inf, -np.inf
    for k in range(K):
        b_k = _quantile(p_eff, grid, d_alpha[k], tail=True)
        c[k] = b_k / math.sqrt(t[k]); b_prev = b_k
        a_k = _quantile(p_fut, grid, d_beta[k], tail=False)
        a[k] = a_k / math.sqrt(t[k]); a_prev = a_k
        if k < K - 1:
            d = dt[k + 1]
            p_eff = _convolve(p_eff * (grid <= b_k), grid, 0.0, math.sqrt(d))
            p_fut = _convolve(p_fut * ((grid > a_k) & (grid <= b_k)), grid, theta * d, math.sqrt(d))
    return c, a


c_bound, a_bound = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde)
boundaries = pd.DataFrame({
    "взгляд": np.arange(1, design.K + 1),
    "t_k": design.t,
    "c_k (efficacy)": np.round(c_bound, 3),
    "a_k (futility)": np.round(a_bound, 3),
})
boundaries


In [ ]:
tt = np.asarray(design.t)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, sp, title in [(axes[0], alpha_spending(tt, design.alpha, "obf"), "α-spending (O'Brien–Fleming)"),
                       (axes[1], beta_spending(tt, design.beta, "obf"), "β-spending (ρ=3)")]:
    ax.plot(tt, sp, marker="o", color=C_EFF if "α" in title else C_FUT)
    ax.fill_between(tt, sp, alpha=0.2, color=C_EFF if "α" in title else C_FUT)
    ax.set_title(title); ax.set_xlabel("t"); ax.set_ylabel("накопленная ошибка")
    ax.axhline(sp[-1], ls="--", color="black", lw=0.8)
fig.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(tt, c_bound, color=C_EFF, lw=2, marker="o", label="efficacy-граница c_k")
ax.plot(tt, a_bound, color=C_FUT, lw=2, marker="o", label="futility-граница a_k")
ax.fill_between(tt, c_bound, 4, color=C_EFF, alpha=0.10)
ax.fill_between(tt, -4, a_bound, color=C_FUT, alpha=0.10)
ax.fill_between(tt, a_bound, c_bound, color=C_CONT, alpha=0.05)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("доля информации t"); ax.set_ylabel("Z(t)")
ax.set_title("Двухграничный дизайн: зоны остановки по эффективности и бесперспективности")
ax.legend(); plt.show()


### 5.1. Расширение на двустороннюю проверку

В одностороннем тесте futility означает «эффект не догонит MDE в выгодную сторону». Для **двустороннего** теста:

- efficacy: останавливаемся при $|Z_t| > c_k$ (граница симметрична, α делится на два хвоста);
- futility: останавливаемся, когда эффект «зажат» в симметричном коридоре $|Z_t| < a_k$ — то есть величина эффекта слишком мала, чтобы дотянуться до значимости в любую сторону.

Визуально это зеркальная картинка: те же $c_k$ и $a_k$ отражаются ниже нуля, и зона futility — «воронка» вокруг нуля, а зоны efficacy — два симметричных хвоста.


In [ ]:
z_alpha2 = norm.ppf(1 - design.alpha / 2)
theta2 = z_alpha2 + norm.ppf(1 - design.beta)   # дрейф для двустороннего MDE
c2, a2 = group_sequential_boundaries(design.t, design.alpha, design.beta, theta2, fam_alpha="obf", fam_beta="obf")

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(tt, c2, color=C_EFF, lw=2, marker="o", label="± efficacy (двустор.)")
ax.plot(tt, -c2, color=C_EFF, lw=2, marker="o")
ax.plot(tt, a2, color=C_FUT, lw=2, marker="o", label="± futility (двустор.)")
ax.plot(tt, -a2, color=C_FUT, lw=2, marker="o")
ax.fill_between(tt, a2, -a2, color=C_FUT, alpha=0.10)
ax.fill_between(tt, c2, 4, color=C_EFF, alpha=0.10)
ax.fill_between(tt, -4, -c2, color=C_EFF, alpha=0.10)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("доля информации t"); ax.set_ylabel("Z(t)")
ax.set_title("Двусторонний дизайн: futility = коридор |Z| < a_k вокруг нуля")
ax.legend(); plt.show()


## 6. Binding vs non-binding futility

Как futility-граница влияет на ошибку I рода? Два способа «посчитать α»:

- **Binding (обязывающая)**: futility-стоп обязателен, и α считается с учётом того, что при $Z_t < a_k$ мы гарантированно останавливаемся. Если потом проигнорировать стоп — суммарный α вырастет.
- **Non-binding (необязывающая)**: α считается так, *будто futility-границы нет*. Тогда игнорирование futility-стопа не раздувает α — это стандарт и дефолт для продуктовых A/B.

Ниже — демонстрация: суммарный α efficacy-границы в трёх случаях.


In [ ]:
def overall_alpha_upper(c: np.ndarray, t: np.ndarray, lower=None, n_sim: int = 500000):
    """Monte-Carlo оценка P(хотя бы один взгляд Z_k > c_k) под H0 (theta=0)."""
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(0, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    active = np.ones(n_sim, dtype=bool)
    crossed = np.zeros(n_sim, dtype=bool)
    for k in range(len(t)):
        if lower is not None:                      # binding: убираем остановленных по futility
            active &= ~(Z[:, k] < lower[k]) & ~crossed
        crossed |= active & (Z[:, k] > c[k])
        active &= ~crossed
    return crossed.mean()


t = np.asarray(design.t)
alpha_none = overall_alpha_upper(c_bound, t, lower=None)
alpha_nonb = overall_alpha_upper(c_bound, t, lower=np.full(len(t), -np.inf))
alpha_bind = overall_alpha_upper(c_bound, t, lower=a_bound)

res = pd.DataFrame({
    "Сценарий": ["без futility-границы", "non-binding", "binding"],
    "Суммарный α (эффективность)": np.round([alpha_none, alpha_nonb, alpha_bind], 4),
})
res["Отличие от номинального α"] = np.round(res["Суммарный α (эффективность)"] - design.alpha, 4)
res


## 7. Влияние futility на ошибки I/II рода + Monte-Carlo

Главный статистический факт лекции:

- ранняя остановка по futility **не раздувает ошибку I рода** (в отличие от peeking по эффективности): она лишь отбрасывает часть траекторий, которые *могли бы* дойти до отвержения — вероятность отвергнуть при $H_0$ может только уменьшиться;
- но futility **снижает мощность**: часть траекторий, где эффект реально есть, на раннем взгляде выглядит слабо и останавливается зря.

Проверим это симуляцией и измерим «цену» — операционные характеристики: мощность, вероятность futility-стопа под H0/H1 и средний размер выборки (ASN).


In [ ]:
def simulate_design(theta, c, a, t, n_sim=50000):
    """MC группо-последовательного дизайна. Исходы: efficacy / futility / complete."""
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    outcome = np.full(n_sim, "complete")
    stop_frac = np.ones(n_sim)
    for k in range(len(t)):
        active = outcome == "complete"
        if not active.any():
            break
        eff = active & (Z[:, k] > c[k])
        fut = active & (Z[:, k] < a[k])
        outcome[eff] = "efficacy"; outcome[fut] = "futility"
        stop_frac[eff] = t[k]; stop_frac[fut] = t[k]
    return outcome, stop_frac


def oc(theta, c, a, t, n_sim=50000):
    out, stop = simulate_design(theta, c, a, t, n_sim)
    p_eff = (out == "efficacy").mean()
    p_fut = (out == "futility").mean()
    p_comp = (out == "complete").mean()
    asn = stop.mean()
    return p_eff, p_fut, p_comp, asn


t = np.asarray(design.t)
p_eff, p_fut, p_comp, asn = oc(0.0, c_bound, a_bound, t)
print(f"Под H0 (theta=0):      FPR={p_eff:.4f}  futility-stop={p_fut:.3f}  complete={p_comp:.3f}  E[N]/Nmax={asn:.3f}")
p_eff, p_fut, p_comp, asn = oc(design.theta_mde, c_bound, a_bound, t)
print(f"Под H1 (theta=MDE):    power={p_eff:.4f}  futility-stop={p_fut:.3f}  complete={p_comp:.3f}  E[N]/Nmax={asn:.3f}")


In [ ]:
thetas = np.linspace(-0.2, 3.0, 17)
rows = [oc(th, c_bound, a_bound, t, 30000) for th in thetas]
oc_df = pd.DataFrame(rows, columns=["power", "futility_stop", "complete", "asn"])
oc_df["theta"] = thetas

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(thetas, oc_df["power"], color=C_EFF, lw=2, label="power (efficacy)")
ax.plot(thetas, oc_df["futility_stop"], color=C_FUT, lw=2, label="P(futility stop)")
ax.plot(thetas, oc_df["complete"], color=C_CONT, lw=2, label="P(дошли до конца)")
ax.axvline(design.theta_mde, color="black", ls="--", lw=1, label="θ_MDE")
ax.set_xlabel("истинный дрейф θ"); ax.set_ylabel("вероятность исхода")
ax.set_title("Операционные характеристики: сумма трёх кривых = 1")
ax.legend(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, theta, title in [(axes[0], 0.0, "H0: почти всё уходит в futility"),
                          (axes[1], design.theta_mde, "H1: большинство — в efficacy")]:
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(300, len(t))), axis=1)
    Z = B / np.sqrt(t)
    for i in range(300):
        col = C_CONT
        for k in range(len(t)):
            if Z[i, k] > c_bound[k]:
                col = C_EFF; break
            if Z[i, k] < a_bound[k]:
                col = C_FUT; break
        ax.plot(t, Z[i], color=col, alpha=0.35, lw=0.8)
    ax.plot(t, c_bound, color=C_EFF, lw=2)
    ax.plot(t, a_bound, color=C_FUT, lw=2)
    ax.set_title(title); ax.set_xlabel("t")
axes[0].set_ylabel("Z(t)")
fig.suptitle("Ансамбль траекторий, раскрашенный по исходу")
fig.tight_layout(); plt.show()


## 8. Практические рекомендации и связь с α-spending

Чек-лист дизайнера futility-анализа:

1. **Тайминг взглядов** — futility-взгляды ставьте не раньше 30–50% данных: ранние данные слишком шумные.
2. **Порог** — CP/PPoS 10–20% типичен; агрессивнее — заметная потеря мощности, мягче — нет экономии.
3. **Предположение об эффекте** — CP под $H_0$ (консервативно) / под трендом (оптимистично) / под MDE (дизайновый ориентир); PPoS — когда есть явный prior.
4. **Non-binding** — дефолт для продуктовых A/B.
5. **Полный дизайн** = α-spending (efficacy-граница) + β-spending (futility-граница) — две стороны одной монеты на оси $t$.

Интерактив ниже собирает всё вместе: крутите порог CP, число взглядов и формы спендинга — и смотрите, как меняются границы и операционные характеристики.


In [ ]:
def design_full(alpha=0.05, beta=0.20, K=5, fam_alpha="obf", fam_beta="obf", cp_threshold=0.15):
    t = np.linspace(0.3, 1.0, K)
    c, a = group_sequential_boundaries(t, alpha, beta, None, fam_alpha, fam_beta)
    p0 = oc(0.0, c, a, t, 20000)
    p1 = oc(norm.ppf(1 - alpha) + norm.ppf(1 - beta), c, a, t, 20000)
    return t, c, a, p0, p1


def plot_design_dashboard(K, fam_alpha, fam_beta, cp_threshold):
    t, c, a, p0, p1 = design_full(0.05, 0.20, K, fam_alpha, fam_beta, cp_threshold)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(t, c, color=C_EFF, lw=2, marker="o", label="efficacy")
    axes[0].plot(t, a, color=C_FUT, lw=2, marker="o", label="futility")
    axes[0].fill_between(t, a, c, color=C_CONT, alpha=0.05)
    axes[0].set_title("Границы"); axes[0].set_xlabel("t"); axes[0].legend()
    axes[1].bar(["FPR (H0)", "power (H1)", "futility (H1)"],
                [p0[0], p1[0], p1[1]], color=[C_CONT, C_EFF, C_FUT])
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"OC: E[N]/Nmax = {p1[3]:.2f} (под H1)")
    fig.tight_layout(); plt.show()

interact(plot_design_dashboard,
         K=IntSlider(value=3, min=2, max=8, step=1, description="взглядов K"),
         fam_alpha=Dropdown(options=["obf", "pocock"], value="obf", description="α-spending"),
         fam_beta=Dropdown(options=["obf", "pocock"], value="obf", description="β-spending"),
         cp_threshold=FloatSlider(value=0.15, min=0.05, max=0.40, step=0.05, description="порог CP"))


## 9. Итоги

1. **Futility не раздувает ошибку I рода** — в отличие от peeking по эффективности; она лишь снижает мощность.
2. **CP** отвечает «достигнем ли значимости при фиксированном θ», **PPoS** усредняет по неопределённости θ; CP — частный случай PPoS.
3. **Stochastic curtailment** — общая рамка; её инкарнации — CP и PPoS.
4. **β-spending — зеркало α-spending**: даёт априорную futility-границу с контролем ошибки II рода.
5. **Non-binding** — безопасный дефолт; тайминг/порог/prior определяют компромисс «мощность ↔ экономия трафика».

| | Conditional Power | PPoS | β-spending |
|---|---|---|---|
| Философия | частотная | байесовская | частотная (дизайн) |
| Предположение об эффекте | фиксированное θ | prior θ | θ_MDE |
| Граница | данные-зависимая | данные-зависимая | фиксирована заранее |
| Что контролирует | правило «стоп» | правило «стоп» | ошибку II рода |
| Когда | быстро оценить шансы | учесть неопределённость | спроектировать тест |
